<div class="alert alert-block alert-info" style="margin-top: 20px">
    <center>
    <h1 align=center><font size = 5>Multiple Dual Interactive Charts<br><br>Time Use in Canada and Vietnam<br><br>By Thuc Dao (Thomas Dao)</font></h1>
    </center>
</div>

### Install libraries (run once)

Note: Do NOT use sas7bdat Python package to read .sas7bdat files.  
Reason: It only works with uncompressed .sas7bdat files and will fail on modern compressed files from SAS 9.4+ (like most StatCan data).  
Here pyreadstat package is used instead.

In [ ]:
# conda is a system package manager
# pip is a Python package manager

# Create a new conda virtual environment
!conda create -n viz_env python=3.10 -y

# Install packages into the virtual environment from conda channels
!conda install -n viz_env -c conda-forge \
    jupyterlab \
    ipykernel \
    pyreadstat \
    pandas \
    numpy \
    -y

# Make the environment appear in Jupyter kernels
!conda run -n viz_env \
    python -m ipykernel install \
    --user \
    --name=viz_env \
    --display-name "Python (viz_env)"

<div class="alert alert-block alert-warning">
<b>Change the environment inside the Jupyter Notebook, by selecting in the menu:<br>  
Kernel → Change Kernel → Python (viz_env)</b>
</div>

In [ ]:
# Check the current kernel
import sys
print(sys.executable)

In [ ]:
# Install packages using pip inside the conda environment
!conda run -n viz_env \
    pip install \
        dash \
        plotly \
        dash-bootstrap-components

### Import libraries

In [1]:
import pyreadstat
import os
import pandas as pd
import numpy as np
import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.io as pio
from datetime import datetime
from time import gmtime, strftime

### Load Vietnamese data

In [2]:
# Input path
vn_file = r'Data sources\VNM_2022_TUS_v01_M\Data\4_diary_main.csv'

print("Loading Vietnamese data from CSV file...")
df_vn = pd.read_csv(vn_file, usecols=["ID", "age", "gender", "BEGIN", "Q401"], encoding='latin-1')

# Sample path
vn_sample_file = r'input\VN_time_use_EXTRACTED.csv'

# Save a sample CSV
df_vn.to_csv(vn_sample_file, index=False, encoding='utf-8-sig')

print("✅ VN data loaded.")

Loading Vietnamese data from CSV file...
✅ VN data loaded.


In [68]:
# Show any 10 rows
df_vn.sample(10, random_state=42)

,ID,age,gender,BEGIN,Q401
5649,12680955108533,42,2,440,1502
26587,252270789700326,40,2,1130,601
4250,10180058900130,28,1,810,507
66682,525492191901822,34,2,400,602
20138,201830621700235,29,1,560,201
55265,404321713402019,50,1,1020,601
91223,777522667402105,34,2,840,1503
11567,141220392800613,21,2,1040,601
60781,444581910500404,46,2,370,601
69287,565722257000227,39,2,920,1001


In [69]:
# Show basic information
df_vn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118933 entries, 0 to 118932
Data columns (total 5 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   ID      118933 non-null  int64
 1   age     118933 non-null  int64
 2   gender  118933 non-null  int64
 3   BEGIN   118933 non-null  int64
 4   Q401    118933 non-null  int64
dtypes: int64(5)
memory usage: 4.5 MB


### Load Canadian data with pyreadstat (at least 5x faster than pandas.read_sas)

In [3]:
# Input path
ca_episode_file = r'Data sources\CAN_TU_ET_2022\Data_Données\TU_ET_2022_Episode_PUMF.sas7bdat'
ca_main_file = r'Data sources\CAN_TU_ET_2022\Data_Données\TU_ET_2022_Main_Principal_PUMF.sas7bdat'

print("Loading Canadian data from SAS files...")
df_ca_episode, meta_ca_episode = pyreadstat.read_sas7bdat(ca_episode_file, usecols=["PUMFID", "STARTMIN", "TUI_01"])
df_ca_main, meta_ca_main = pyreadstat.read_sas7bdat(ca_main_file, usecols=["PUMFID", "GENDER2", "AGEGR10"])

# Left join the episode file with the main file on the PUMFID
df_ca = pd.merge(df_ca_episode, df_ca_main, on="PUMFID", how="left")

# Sample path
ca_sample_file = r'input\CA_time_use_EXTRACTED_JOINED.csv'

# Save a sample CSV
df_ca.to_csv(ca_sample_file, index=False, encoding='utf-8-sig')

print("✅ CA data loaded.")

Loading Canadian data from SAS files...
✅ CA data loaded.


In [71]:
# Show any 10 rows
df_ca.sample(10, random_state=42)

,PUMFID,STARTMIN,TUI_01,AGEGR10,GENDER2
80392,6895.0,470.0,207.0,6.0,2.0
137860,11121.0,820.0,261.0,4.0,2.0
70582,6185.0,765.0,299.0,6.0,2.0
61289,5502.0,1315.0,1201.0,7.0,2.0
41805,4071.0,510.0,407.0,5.0,2.0
27607,3024.0,720.0,151.0,5.0,1.0
4903,1362.0,810.0,1203.0,5.0,1.0
26524,2947.0,1200.0,1001.0,4.0,1.0
75530,6540.0,1400.0,101.0,6.0,1.0
117489,9616.0,510.0,407.0,4.0,2.0


In [72]:
# Show basic information
df_ca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168078 entries, 0 to 168077
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   PUMFID    168078 non-null  float64
 1   STARTMIN  168078 non-null  float64
 2   TUI_01    168078 non-null  float64
 3   AGEGR10   168078 non-null  float64
 4   GENDER2   168078 non-null  float64
dtypes: float64(5)
memory usage: 6.4 MB


### Helper Functions

In [73]:
def summarize_activity_data(df):
    """
    Prints summary statistics and top activities from the provided DataFrame.
    """
    print("Total records:", len(df))
    print("Unique individuals:", df['person_id'].nunique())
    print("Time range:", df['date_time'].min(), "to", df['date_time'].max())
    print("\nAll activities ranked by frequency:")
    print(df['activity_group'].value_counts())

In [74]:
def get_safe_filepath(filepath):
    """
    Checks if a file exists. If it does, prompts the user to overwrite or rename.
    """
    while os.path.exists(filepath):
        print(f"\n[!] Warning: The file '{filepath}' already exists.")
        choice = input("Do you want to (o)verwrite it or (r)ename the new file? [o/r]: ").strip().lower()
        
        if choice == 'o':
            print(f"Overwriting '{filepath}'...")
            break # Breaks loop, returns the original filepath to overwrite
        elif choice == 'r':
            new_name = input("Enter the new file name: ").strip()
            # Ensure the new file still goes into the correct directory
            directory = os.path.dirname(filepath)
            filepath = os.path.join(directory, new_name)
            # The loop continues to check if this NEW name also exists
        else:
            print("Invalid choice. Please enter 'o' to overwrite or 'r' to rename.")
            
    return filepath

In [75]:
def get_vn_activity(code):
    code = int(code)
    if code in [101, 102, 199, 201, 202, 299, 301, 302, 399, 401, 402, 499, 502]:
        return "Work"
    elif code in [198, 298, 298, 498, 598, 698, 798, 898, 998, 1098, 1198, 1298, 1398, 1498, 1598]:
        return "Travelling"
    elif code in [501, 504, 505, 506, 507, 508]:
        return "Sell goods or provide services"
    elif code in [601]:
        return "Housework"
    elif code in [602]:
        return "Shopping"
    elif code in [701, 702]:
        return "Caring"
    elif code in [901, 902, 903]:
        return "Education"
    elif code in [1201, 1202, 1203, 1299]:
        return "Active leisure"
    elif code in [1301]:
        return "Sport"
    elif code in [1302, 1399]:
        return "Outdoor activities"
    elif code == 1402:
        return "TV/Online videos"
    elif code == 1404:
        return "Passive leisure"
    elif code == 1501:
        return "Sleeping"
    elif code == 1502:
        return "Eating"
    elif code == 1503:
        return "Personal care"
    elif code == 1506:
        return "Relaxing"
    else:
        return "Others"

### Map Canadian activity codes to Vietnamese-style categories

| Canadian Code(s)          | Description                                        | Vietnamese Equivalent Category        |
| ------------------------- | -------------------------------------------------- | ------------------------------------- |
| 501–507, 599              | Paid work, selling, training, looking for work     | **Work**                              |
| 505                       | Selling of goods or services for pay               | **Sell goods or provide services**    |
| 201–209, 231–241          | Cooking, cleaning, household chores, DIY, repairs  | **Housework**                         |
| 151–154, 404              | Eating or drinking, food-related travel            | **Eating**                            |
| 100–109                   | Sleep, nap, rest                                   | **Sleeping**                          |
| 126–130, 199, 401         | Personal hygiene, medical, personal care           | **Personal care**                     |
| 150                       | Eating/drinking (top category)                     | **Eating**                            |
| 260–264                   | In-person or online shopping, research             | **Shopping**                          |
| 300–307, 350–353, 801–802 | Caring for children/adults                         | **Caring**                            |
| 601–604, 600              | School, homework, learning, tutoring               | **Education**                         |
| 1001–1005, 413            | Exercise, sport, physical activity, related travel | **Sport**                             |
| 1202                      | Watching TV, movies, online videos                 | **TV/Online videos**                  |
| 1201, 1203–1204           | Reading, music, podcasts, general tech use         | **Passive leisure**                   |
| 1105                      | Arts, hobbies, games, crafts, video games          | **Active leisure**                    |
| 1106                      | Outdoor leisure, birdwatching, boating             | **Outdoor activities**                |
| 1301–1303                 | Waiting, doing nothing, thinking                   | **Relaxing**                          |
| 409, 407, 411, 400–416    | Travel for work, shopping, study, care, etc.       | **Travelling**                        |
| 9999, 1304                | Unspecified or other activities                    | **Others**                            |


**Notes:**  
Categories like “Sell food” or “Provide services” don’t map directly from Canadian codes  
but can be inferred from specific self-employment or home-business entries (505).  
These can be differentiated only with further context.

In [76]:
ca_to_vn = {
    "Work": [501, 502, 503, 504, 505, 506, 507, 599],
    "Sell goods or provide services": [505],
    "Housework": list(range(201, 210)) + list(range(231, 242)),
    "Eating": [150, 151, 152, 153, 154, 404],
    "Sleeping": list(range(100, 110)),
    "Personal care": [126, 127, 128, 129, 130, 199, 401],
    "Shopping": [260, 261, 262, 263, 264],
    "Caring": list(range(300, 308)) + list(range(350, 354)) + [801, 802],
    "Education": [600, 601, 602, 603, 604],
    "Sport": [1001, 1002, 1003, 1004, 1005, 413],
    "TV/Online videos": [1202],
    "Passive leisure": [1201, 1203, 1204],
    "Active leisure": [1105],
    "Outdoor activities": [1106],
    "Relaxing": [1301, 1302, 1303],
    "Travelling": list(range(400, 417)),
    "Others": [9999, 1304]
}

In [77]:
def map_ca_activity(code):
    try:
        if pd.isna(code): return "Others"
        code = int(float(code)) # Handle float codes
        for category, codes in ca_to_vn.items():
            if code in codes:
                return category
    except (ValueError, TypeError):
        pass
    return "Others"

In [78]:
# Define the universal category and order for Canada and Vietnam
category_order = {
    "Sleeping": "Sleeping",
    "Personal hygiene": "Personal care",
    "Eating": "Eating",
    "Housework": "Housework",
    "Caring": "Caring",
    "Work": "Work",
    "Education": "Education",
    "Provide services": "Providing services",
    "Sell food": "Selling food",
    "Shopping": "Shopping",
    "Travelling": "Travelling",
    "Sport": "Sport",
    "Relaxing": "Relaxing",
    "Entertainment": "Entertainment",
    "TV/Online videos": "TV/Online videos",
    "Surf web": "Surfing web",
    "Others": "Others"
}

In [79]:
COLORS = {
    "Sleeping": "#2c3e50", 
    "Personal care": "#34495e", 
    "Eating": "#f1c40f",
    "Housework": "#e67e22", 
    "Caring": "#d35400", 
    "Work": "#e74c3c",
    "Education": "#3498db", 
    "Providing services": "#2980b9", 
    "Selling food": "#1abc9c",
    "Shopping": "#16a085", 
    "Travelling": "#9b59b6", 
    "Sport": "#2ecc71",
    "Relaxing": "#27ae60", 
    "Entertainment": "#95a5a6", 
    "TV/Online videos": "#7f8c8d",
    "Surfing web": "#bdc3c7", 
    "Others": "#ecf0f1"
}

In [80]:
genders = ["Male", "Female"]
age_groups = ["15 to 34 years", "35 to 54 years", "55 to 64 years", "65 years and over"]

def standardize_columns(df, is_canada=False):
    """Applies some transformations to the raw data."""
    df = df.copy()
    
    if is_canada:
        df['activity_group'] = df['activity_group'].apply(map_ca_activity)
        df['gender'] = df['gender'].map({1.0: "Male", 2.0: "Female"})
        ca_age_map = {1.0: "15 to 34 years", 2.0: "15 to 34 years", 
                      3.0: "35 to 54 years", 4.0: "35 to 54 years", 
                      5.0: "55 to 64 years", 
                      6.0: "65 years and over", 7.0: "65 years and over"}
        df['age_group'] = df['age_group'].map(ca_age_map)
    else:
        df['activity_group'] = df['activity_group'].apply(get_vn_activity)
        df['gender'] = df['gender'].map({1: "Male", 2: "Female"})
        df['age_group'] = pd.cut(df['age'], 
                                 bins=[14, 34, 54, 64, 150], 
                                 labels=age_groups)

    # Universal category and order for both Canada and Vietnam
    df['activity_group'] = df['activity_group'].map(category_order)
    return df

### Transform Vietnamese time use data

In [81]:
print("Processing Vietnamese data...")
df_vn = df_vn.rename(columns={"ID": "person_id", "BEGIN": "date_time", "Q401": "activity_group"})
df_vn = df_vn.sort_values(["person_id", "date_time"])
df_vn = df_vn.dropna(subset=['person_id', 'date_time', 'activity_group'])

# Standardize column values
df_vn = standardize_columns(df_vn, is_canada=False)

summarize_activity_data(df_vn)

Processing Vietnamese data...
Total records: 118933
Unique individuals: 6001
Time range: 240 to 1670

All activities ranked by frequency:
activity_group
Housework             17191
Eating                16514
Travelling            16171
Sleeping              15320
Personal care         14419
Work                   8422
Caring                 6570
Relaxing               5160
TV/Online videos       4579
Selling food           4115
Others                 3302
Surfing web            1799
Shopping               1711
Education              1259
Sport                  1213
Providing services      792
Entertainment           396
Name: count, dtype: int64


### Transform Canadian time use data

In [82]:
print("Processing Canadian data...")
df_ca = df_ca.rename(columns={"PUMFID": "person_id", "STARTMIN": "date_time", "TUI_01": "activity_group", "GENDER2": "gender", "AGEGR10": "age_group"})
df_ca = df_ca.sort_values(["person_id", "date_time"])
df_ca = df_ca.dropna(subset=['person_id', 'date_time', 'activity_group'])

# Standardize column values
df_ca = standardize_columns(df_ca, is_canada=True)

summarize_activity_data(df_ca)

Processing Canadian data...
Total records: 168078
Unique individuals: 12336
Time range: 240.0 to 1675.0

All activities ranked by frequency:
activity_group
Sleeping            27313
Housework           25569
Eating              24198
Travelling          19175
Personal care       15911
Others              11526
TV/Online videos    11328
Work                 8458
Surfing web          6412
Caring               4964
Sport                4530
Shopping             3508
Relaxing             2318
Entertainment        1649
Education            1219
Name: count, dtype: int64


### Convert episodic data to a minute-by-minute population percentage matrix

In [83]:
def transform_to_cross_sectional(df, id_col, time_col, act_col):
    """Converts episodic data to a minute-by-minute population percentage matrix."""
    if df.empty:
        # Return an empty matrix safely if no data matches the filter
        dates = pd.date_range("2022-01-01 04:00:00", periods=144, freq="10min")
        return pd.DataFrame(0, index=dates, columns=category_order.values())
        
    df = df.copy()

    # Drop any rows where the time column is NaN/Null to prevent crash
    df = df.dropna(subset=[time_col])

    # Safely cast the time in minute to integer 
    df['minute'] = pd.to_numeric(df[time_col], errors='coerce').fillna(0).astype(int)

    # Process activity states
    df['state'] = df[act_col].fillna("Others")
    df = df.sort_values([id_col, 'minute'])

    # Adjust bins to match actual data range of time
    # Data starts at 4:00 AM (240 mins) and ends at 4:00 AM next day (1680 mins)
    time_bins = np.arange(240, 1690, 10) # 10-minute intervals for performance
    df['time_bin'] = pd.cut(df['minute'], bins=time_bins, labels=time_bins[:-1], right=False)

    # Pivot and forward fill
    pivot = df.drop_duplicates(subset=[id_col, 'time_bin'], keep='last')
    pivot = pivot.pivot(index='time_bin', columns=id_col, values='state')
    pivot = pivot.ffill().bfill() 

    # Aggregation
    activity_counts = pivot.apply(pd.Series.value_counts, axis=1).fillna(0)
    activity_pct = activity_counts.div(activity_counts.sum(axis=1), axis=0) * 100

    # Ensure all categories are present
    for cat in category_order.values():
        if cat not in activity_pct.columns: 
            activity_pct[cat] = 0.0
            
    activity_pct = activity_pct[sorted(activity_pct.columns)]

    # Create final Datetime Index for Plotly
    base_time = pd.Timestamp("2022-01-01 00:00:00")
    # Adding 240 mins becomes 04:00:00, adding 1680 mins becomes 04:00:00 the next day
    activity_pct.index = [base_time + pd.Timedelta(minutes=int(m)) for m in activity_pct.index]
    
    return activity_pct

In [84]:
# Pre-calculate all slices
print("Calculating cross-sectional matrices...")
matrices = {
    "Overview": {
        "CA": transform_to_cross_sectional(df_ca, "person_id", "date_time", "activity_group"),
        "VN": transform_to_cross_sectional(df_vn, "person_id", "date_time", "activity_group")
    }
}

# Add Gender slices
for g in genders:
    matrices[f"Gender_{g}"] = {
        "CA": transform_to_cross_sectional(df_ca[df_ca['gender'] == g], "person_id", "date_time", "activity_group"),
        "VN": transform_to_cross_sectional(df_vn[df_vn['gender'] == g], "person_id", "date_time", "activity_group")
    }

# Add Age slices
for a in age_groups:
    matrices[f"Age_{a}"] = {
        "CA": transform_to_cross_sectional(df_ca[df_ca['age_group'] == a], "person_id", "date_time", "activity_group"),
        "VN": transform_to_cross_sectional(df_vn[df_vn['age_group'] == a], "person_id", "date_time", "activity_group")
    }

print("Calculation completed.")

Calculating cross-sectional matrices...
Calculation completed.


### Generate charts

In [89]:
def build_traces(df, country_prefix, visible=True):
    """Builds the stacked area traces for a single dataframe."""
    traces = []
    for col in df.columns:
        traces.append(go.Scatter(
            x=df.index, y=df[col],
            name=str(col), # values in the activity_group column
            mode='lines',
            line=dict(width=0.5, color=COLORS.get(col, "#000")),
            stackgroup=f'stack_{country_prefix}', # Separate stack groups for each chart
            fillcolor=COLORS.get(col, "#000"),
            xaxis=f'x{country_prefix}',
            yaxis=f'y{country_prefix}',
            visible=visible,
            hovertemplate='%{y:.1f}%<extra></extra>',
            legendgroup=col, # Links legends between charts
            showlegend=(country_prefix == 1) # Only show legend once
        ))
    return traces

In [92]:
def export_standalone_html(title, filename, keys_to_include, dropdown_labels=None):
    """
    Creates a side-by-side subplot chart with an optional dropdown, 
    and exports it as a standalone HTML file.
    """
    print(f"Generating {filename}...")
    fig = go.Figure()

    num_categories = len(keys_to_include)
    total_traces_per_view = len(category_order) * 2 # 17 cats * 2 countries = 34

    # Add all traces for all views
    for i, key in enumerate(keys_to_include):
        is_first_view = (i == 0)
        # Left Chart: Canada (prefix=1)
        traces_ca = build_traces(matrices[key]["CA"], 1, visible=is_first_view)
        # Right Chart: Vietnam (prefix=2)
        traces_vn = build_traces(matrices[key]["VN"], 2, visible=is_first_view)
        
        fig.add_traces(traces_ca + traces_vn)

    # Setup layout
    layout_args = dict(
        title=title,
        plot_bgcolor="white",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
        # Subplot 1 (Left - CA)
        xaxis=dict(
            domain=[0, 0.48], type="date", tickformat="%H:%M", 
            title="Canada", showgrid=False,
            rangeslider=dict(visible=True, thickness=0.1) # <-- Added rangeslider
        ),
        yaxis=dict(range=[0, 100], showgrid=True, title="% Population"),
        
        # Subplot 2 (Right - VN)
        xaxis2=dict(
            domain=[0.52, 1], type="date", tickformat="%H:%M", 
            title="Vietnam", showgrid=False,
            rangeslider=dict(visible=True, thickness=0.1) # <-- Added rangeslider
        ),
        yaxis2=dict(range=[0, 100], showgrid=True, anchor="x2")
    )

    # Add Dropdown if needed
    if dropdown_labels:
        buttons = []
        for i, label in enumerate(dropdown_labels):
            # Create a boolean list where only the traces for the current view are True
            visibility = [False] * (num_categories * total_traces_per_view)
            start_idx = i * total_traces_per_view
            end_idx = start_idx + total_traces_per_view
            visibility[start_idx:end_idx] = [True] * total_traces_per_view
            
            buttons.append(dict(
                label=label,
                method="update",
                args=[{"visible": visibility}, {"title": f"{title} - {label}"}]
            ))
            
        layout_args["updatemenus"] = [dict(
            active=0,
            buttons=buttons,
            x=0.0, xanchor="left", y=1.15, yanchor="top",
            pad={"r": 10, "t": 10},
            showactive=True,
        )]

    fig.update_layout(**layout_args)

    # Extract the HTML source code as a string
    html_string = pio.to_html(fig, include_plotlyjs='cdn', full_html=True)

    # THE SECRET WEAPON: JavaScript snippet to automatically stack Plotly coordinates on mobile devices
    responsive_js = """
    <script>
        // Wait 500ms for Plotly to finish rendering the initial framework
        setTimeout(function() {
            // Get the DOM element of the newly created Plotly graph
            var graphDiv = document.getElementsByClassName('plotly-graph-div')[0];
            
            function makeResponsive() {
                // Check if the screen width is less than or equal to 768px (Mobile breakpoint)
                var isMobile = window.innerWidth <= 768;
                
                // Reconfigure layout coordinates:
                // If Mobile: X-axes take 100% width. Y-axes split top/bottom.
                // If Desktop: X-axes split left/right. Y-axes take 100% height.
                var layoutUpdate = {
                    'xaxis.domain': isMobile ? [0, 1] : [0, 0.48],
                    'yaxis.domain': isMobile ? [0.55, 1] : [0, 1],
                    'xaxis2.domain': isMobile ? [0, 1] : [0.52, 1],
                    'yaxis2.domain': isMobile ? [0, 0.45] : [0, 1]
                };
                
                // Instruct Plotly to seamlessly redraw the layout with new coordinates
                Plotly.relayout(graphDiv, layoutUpdate);
            }

            // Trigger the responsive function immediately upon file load
            makeResponsive();

            // Event listener: Whenever the user rotates their phone 
            // or resizes the browser window, automatically re-run the layout function
            window.addEventListener('resize', makeResponsive);
        }, 500);
    </script>
    </body>
    """

    # Inject the JavaScript snippet right before the closing </body> tag of the HTML file
    final_html = html_string.replace('</body>', responsive_js)

    # Save the final content as a static HTML file
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(final_html)
        
    print(f"Successfully exported {filename} (Mobile Responsive Integrated).")

### Export charts to HTML

In [93]:
# Create the 'output' folder if it doesn't already exist
os.makedirs("output", exist_ok=True)

# Check file existence for charts
overview_filepath = get_safe_filepath("output/Overview_Charts.html")
gender_filepath = get_safe_filepath("output/Gender_Charts.html")
age_filepath = get_safe_filepath("output/Age_Charts.html")

# Extract JUST the filenames (without the 'output/' folder path)
overview_filename = os.path.basename(overview_filepath)
gender_filename = os.path.basename(gender_filepath)
age_filename = os.path.basename(age_filepath)

# 1. Overview
export_standalone_html(
    "Time Use Overview", 
    overview_filepath, 
    ["Overview"])

# 2. By Gender (With Dropdown)
export_standalone_html(
    "Time Use by Gender", 
    gender_filepath, 
    ["Gender_Male", "Gender_Female"], 
    dropdown_labels=genders
)

# 3. By Age (With Dropdown)
export_standalone_html(
    "Time Use by Age Group", 
    age_filepath, 
    [f"Age_{a}" for a in age_groups], 
    dropdown_labels=age_groups
)

print("Export complete! Check your folder for the HTML files.")

Generating output/Overview_Charts.html...
Successfully exported output/Overview_Charts.html (Mobile Responsive Integrated).
Generating output/Gender_Charts.html...
Successfully exported output/Gender_Charts.html (Mobile Responsive Integrated).
Generating output/Age_Charts.html...
Successfully exported output/Age_Charts.html (Mobile Responsive Integrated).
Export complete! Check your folder for the HTML files.
